In [1]:
!pip install finlight-client

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [13]:
!pip install spacy

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 8.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 653.2/653.2 kB 8.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.1/741.1 kB 6.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.1 MB/s  0:00:00
  Attempting uninstall: typer╺━━━━━━━━━━━━━━━━━━  9/17 [srsly]athlib]
    Found existing installation: typer 0.13.1━━━━━━━━━━━━━━━━━  9/17 [srsly]
    Uninstalling typer-0.13.1:╺━━━━━━━━━━━━━━━━━━  9/17 [srsly]
      Successfully uninstalled typer-0.13.1m━━━━━━━━━━━━━━━━━━  9/17 [srsly]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [spacy]m16/17 [spacy]]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [33]:
!pip install spacy-entity-linker

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for spacy-entity-linker: filename=spacy_entity_linker-1.0.3-py3-none-any.whl size=14453 sha256=e195d9f4d9ed48c9870c7200ae25744d45ea32e350348be0831281b16d240f02
  Stored in directory: /Users/anjanasajith/Library/Caches/pip/wheels/7a/aa/9d/26bd92087fa7710ffcc7415c2ae41120e68aa9f7a8844ba9f9
Successfully built spacy-entity-linker

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [14]:
import json
import spacy
from finlight_client import FinlightApi, ApiConfig
from finlight_client.models import GetArticlesParams

In [28]:


client = FinlightApi(
        config=ApiConfig(
            api_key="sk_ef1ab2502784c29c6c78359ee5a3f4d46f5fcad8b82fb5c989422b8fdd180ad1"
        )
    )

params = GetArticlesParams(tickers=["NVDA", "TSM", "MSFT", "AVGO", "ASML", "AMD", "INTC", "AAPL"], # Multiple companies
    query='("data center" OR "AI infrastructure" OR "foundry" OR "semiconductor")', 
    language="en",
    pageSize=50)
articles = client.articles.fetch_articles(params=params)

In [21]:
if articles.articles:
    sample_article = articles.articles[0]
    
    print(sample_article.__dict__.keys())

dict_keys(['link', 'title', 'publishDate', 'source', 'language', 'summary', 'images', 'createdAt', 'sentiment', 'confidence', 'content', 'companies'])


In [29]:
articles_list = []
for article in articles.articles:
    articles_list.append({
        "title": article.title,
        "source": article.source,
        "content": article.content or article.summary, # Fallback to summary
        "date": str(article.publishDate),
        "sentiment": article.sentiment,
        "companies": article.companies
    })


articles_json = json.dumps(articles_list, indent=4)
print(f"Fetched {len(articles_list)} articles.")

Fetched 50 articles.


In [30]:
articles_list

[{'title': 'Nvidia Stock Investors Just Got Good News From Amazon, Google, Meta Platforms, and Microsoft',
  'source': 'finance.yahoo.com',
  'content': 'Hyperscalers are likely to spend much more on AI infrastructure in 2026 than Wall Street initially estimated.',
  'date': '2026-02-14 09:32:00+00:00',
  'sentiment': None,
  'companies': None},
 {'title': 'Nvidia-Leased Data Center Wraps Up In-Demand $3.8B Bond',
  'source': 'www.bloomberg.com',
  'content': "A data center project expected to be leased by\xa0Nvidia Corp.\xa0sold $3.8 billion of junk bonds on Friday after receiving about $14 billion of orders from investors, an indication that lenders are eager to continue funding the buildout of artificial-intelligence infrastructure.\xa0Bloomberg's Aaron Weinman reports. (Source: Bloomberg)",
  'date': '2026-02-13 23:21:44+00:00',
  'sentiment': None,
  'companies': None},
 {'title': 'Competition Is Heating Up for Micron. Should You Buy, Sell, or Hold MU Stock Now?',
  'source': 'fin

In [31]:
# NLP Processing (Tokenization, Segmentation, NER)
nlp = spacy.load("en_core_web_sm")

In [32]:
for art in articles_list:
    print(f"\n--- Processing: {art['title']} ---")
    
    # Check if content exists
    text = art['content']
    if not text:
        continue
        
    doc = nlp(text)

    # A. Sentence Segmentation
    print(f"Sentences found: {len(list(doc.sents))}")

    # B. Tokenization & Stop-word Removal (First 10 non-stop words)
    tokens = [token.text for token in doc if not token.is_stop and not token.is_punct]
    print(f"Sample Tokens: {tokens[:10]}")

    # C. Named Entity Recognition (NER)
    print("Entities:")
    for ent in doc.ents:
        print(f" - {ent.text} ({ent.label_})")


--- Processing: Nvidia Stock Investors Just Got Good News From Amazon, Google, Meta Platforms, and Microsoft ---
Sentences found: 1
Sample Tokens: ['Hyperscalers', 'likely', 'spend', 'AI', 'infrastructure', '2026', 'Wall', 'Street', 'initially', 'estimated']
Entities:
 - AI (GPE)
 - 2026 (DATE)

--- Processing: Nvidia-Leased Data Center Wraps Up In-Demand $3.8B Bond ---
Sentences found: 3
Sample Tokens: ['data', 'center', 'project', 'expected', 'leased', '\xa0', 'Nvidia', 'Corp.', '\xa0', 'sold']
Entities:
 - Nvidia Corp. (ORG)
 - $3.8 billion (MONEY)
 - Friday (DATE)
 - about $14 billion (MONEY)
 - Bloomberg (PERSON)
 - Aaron Weinman (PERSON)
 - Bloomberg (PERSON)

--- Processing: Competition Is Heating Up for Micron. Should You Buy, Sell, or Hold MU Stock Now? ---
Sentences found: 1
Sample Tokens: ['Samsung', 'advances', 'gen', 'AI', 'memory', 'Micron', 'leadership', 'faces', 'crucial', 'test']
Entities:
 - Samsung (ORG)
 - AI (GPE)
 - Micron (ORG)

--- Processing: 2 AI Stocks That 

In [34]:
nlp.add_pipe("entity_linker", last=True)

for art in articles_list:
    doc = nlp(art['content'])
    
    # Access linked entities
    for entity in doc._.linkedEntities:
        # This gives you a unique identifier (URL/ID) and the formal name
        print(f"Entity: {entity.get_label()} | ID: {entity.get_id()} | URL: {entity.get_url()}")
        print(f"Description: {entity.get_description()}")

ValueError: [E139] Knowledge base for component 'entity_linker' is empty.

In [36]:
!pip install rapidfuzz


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.4 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [37]:
from rapidfuzz import process, utils

master_suppliers = ["Apple Inc.", "Samsung Electronics", "Taiwan Semiconductor Manufacturing Co."]

def link_to_master(detected_org, master_list):
    # Extract the best match from your master list
    result = process.extractOne(detected_org, master_list, processor=utils.default_process)
    
    # If the similarity score is > 85%, assume they are the same
    if result and result[1] > 85:
        return result[0]
    return detected_org # Return original if no confident match

# Inside your loop:
for ent in doc.ents:
    if ent.label_ == "ORG":
        linked_name = link_to_master(ent.text, master_suppliers)
        print(f"Original: {ent.text} -> Linked: {linked_name}")

Original: Investing.com -> Linked: Investing.com
Original: Nvidia Corp. -> Linked: Nvidia Corp.
